# 02 - Joins, unions, nested data, and Gold modeling

Prerequisites / execution context: Run 00 and 01. Replaces Gold exercise outputs.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R20: DataFrame joins](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/join); [R21: DataFrame union](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/union); [R22: DataFrame unionByName](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/unionByName); [R35: Materialized views](https://docs.databricks.com/aws/en/ldp/concepts/materialized-views); [R36: Streaming tables](https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Predict join counts before running
Use the latest orders and a dimension with unique customer IDs. Expected: inner 2, left 4, anti 2, cross 12. A left join preserves unmatched orders.


In [ ]:
o=spark.table(f"{SILVER}.orders")
c=spark.table(f"{SILVER}.customers")
inner=o.join(c,"customer_id","inner")
left=o.join(F.broadcast(c),["customer_id"],"left")
anti=o.join(c,"customer_id","left_anti")
assert (inner.count(),left.count(),anti.count(),o.crossJoin(c).count())==(2,4,2,12)
anti.select("order_id","customer_id").show()
# Filtering a right-side column after a left join removes the unmatched rows.
assert left.filter("country = 'IT'").count()==1


## 2. Expose dimension fanout and a multiple-key join
This mirrors row-count reconciliation during migrations. A broadcast hint changes the physical strategy, not the intended rows. Official Databricks documentation: [R20: DataFrame joins](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/join).


In [ ]:
dup_c=c.unionByName(c.filter("customer_id='00035'"))
assert o.join(dup_c,"customer_id","left").count()==5
fact=spark.createDataFrame([("00035","IT",1),("00035","FR",2)],"customer_id STRING, country STRING, n INT")
assert fact.join(c,["customer_id","country"],"inner").count()==1


## 3. SQL UNION versus DataFrame union
Predict three versus two rows; use unionByName when columns have moved. Official Databricks documentation: [R21: DataFrame union](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/union); [R22: DataFrame unionByName](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/unionByName).


In [ ]:
a=spark.createDataFrame([(1,"x")],"id INT, label STRING")
b=spark.createDataFrame([("x",1),("y",2)],"label STRING, id INT")
u=a.unionByName(b)
assert u.count()==3 and u.distinct().count()==2
a.createOrReplaceTempView("union_a"); b.createOrReplaceTempView("union_b")
assert spark.sql("SELECT id,label FROM union_a UNION SELECT id,label FROM union_b").count()==2
assert spark.sql("SELECT id,label FROM union_a UNION ALL SELECT id,label FROM union_b").count()==3


## 4. Arrays and strings
Compare explode with explode_outer on the empty tags array. Use built-in expressions to rename, split, add, and drop columns. Official Databricks documentation: [R19: PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics).


In [ ]:
assert o.select("order_id",F.explode("tags").alias("tag")).count()==3
assert o.select("order_id",F.explode_outer("tags").alias("tag")).count()==4
(o.withColumn("order_parts",F.split("order_id","o"))
 .withColumnRenamed("seq","source_sequence").drop("tags").show())


## 5. Publish Gold at a declared grain
The output grain is one country bucket. Unknown includes o3 and o6; revenue should be FR 50, IT 120, UNKNOWN 50, total 220. This lab uses a regular table and view; lab 07 builds a materialized view.


In [ ]:
gold=(left.withColumn("country",F.coalesce("country",F.lit("UNKNOWN")))
 .groupBy("country").agg(F.count("*").alias("orders"),F.sum("amount").alias("revenue")))
gold.write.mode("overwrite").saveAsTable(f"{GOLD}.revenue")
spark.sql(f"CREATE OR REPLACE VIEW {GOLD}.revenue_view AS SELECT * FROM {GOLD}.revenue")
assert str(gold.agg(F.sum("revenue")).first()[0])=="220.00"
gold.orderBy("country").show()


## Challenge
Explain why aggregating after exploding tags can overcount revenue. Solution: explosion changes the row grain; aggregate at the original order grain or allocate the amount intentionally.
